# 第 8 章 · 记忆（ADK：用工具改 state）

`tool_context.state` 是**可写**的——工具是修改会话 state 的正规入口。

本例手工构造一个 `ToolContext`，直接调用工具，**不经过模型**。

kernel 选 `Python (agentic-adk)`。三条写 state 的路、前缀与数据表的关系见 `../../my_notes/ch08_memory.md`。


In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）
from common import quiet_adk_sync_warnings, require_adk
quiet_adk_sync_warnings()
require_adk()      # 【适配】内核选错时给出可操作提示

import time

from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event
from google.adk.sessions import InMemorySessionService
from google.adk.tools.tool_context import ToolContext


def log_user_login(tool_context: ToolContext) -> dict:
    """Updates the session state upon a user login event.

    Args:
        tool_context: Automatically provided by ADK, gives access to session state.

    Returns:
        A dictionary confirming the action was successful.
    """
    state = tool_context.state

    login_count = state.get("user:login_count", 0) + 1
    state["user:login_count"] = login_count
    state["task_status"] = "active"
    state["user:last_login_ts"] = time.time()
    state["temp:validation_needed"] = True

    print("State updated from within the `log_user_login` tool.")
    return {"status": "success", "message": f"User login tracked. Total logins: {login_count}."}


# 1. Setup（【适配】2.x 的 create_session 是异步的，内存版提供了 _sync 变体）
session_service = InMemorySessionService()
app_name, user_id, session_id = "state_app_tool", "user3", "session3"
session = session_service.create_session_sync(
    app_name=app_name, user_id=user_id, session_id=session_id,
    state={"user:login_count": 0, "task_status": "idle"})
print(f"Initial state: {session.state}")

# 2. Simulate a tool call（真实应用里由 Runner 注入上下文）
# 【适配】原书给 InvocationContext 传 app_name/user_id/session_id，2.x 只接受
#   session_service / session / invocation_id 三个必填字段
mock_context = ToolContext(invocation_context=InvocationContext(
    session_service=session_service, session=session, invocation_id="inv-1"))

# 3. Execute the tool
log_user_login(mock_context)

# 4. Check the updated state
# 【适配】工具写的是 state_delta，必须提交成事件才落库；不提交则 get_session 读到的仍是旧值
run_async = __import__("common").run_async
run_async(session_service.append_event(session, Event(author="user", actions=mock_context._event_actions)))
updated_session = session_service.get_session_sync(
    app_name=app_name, user_id=user_id, session_id=session_id)
print(f"State after tool execution: {updated_session.state}")


Initial state: {'task_status': 'idle', 'user:login_count': 0}
State updated from within the `log_user_login` tool.
State after tool execution: {'task_status': 'active', 'user:login_count': 1, 'user:last_login_ts': 1794281667.6875944}
